In [0]:
output_path = "/Volumes/workspace/default/data/output/"
dbutils.fs.rm(output_path, True)

In [0]:
orders_path = "/Volumes/workspace/default/data/bookstore/orders/"

In [0]:
from pyspark.sql.functions import *

#### Create some sample parquet datasets

In [0]:
%fs ls /Volumes/workspace/default/data/bookstore/orders/

In [0]:
df1 = (
    spark
    .read
    .parquet(orders_path)
    .withColumn("order_date", to_date(to_timestamp("order_timestamp"), 'y-M-d'))
)

df1.write.parquet(output_path + "parquet1")
df1.write.partitionBy("order_date").parquet(output_path + "parquet2")
df1.write.partitionBy("order_date").parquet(output_path + "parquet3")

In [0]:
df1.printSchema()

In [0]:
display( dbutils.fs.ls(output_path + "parquet3") )

#### Converting Parquet Dataset to Delta using Python

In [0]:
from delta.tables import *

In [0]:
output_path

In [0]:

deltaTable1 = DeltaTable.convertToDelta(spark, f"parquet.`{output_path}/parquet1`")


In [0]:
deltaTable1.toDF().display()

In [0]:
display(dbutils.fs.ls(output_path + "parquet1"))

In [0]:
deltaTable1.history().display()

#### Converting Partitioned Parquet Dataset to Delta using Python

In [0]:
display(dbutils.fs.ls(output_path + "parquet2"))

In [0]:
# converting partitioned parquet dataset to delta format
deltaTable2 = DeltaTable.convertToDelta(spark, f"parquet.`{output_path}/parquet2`", "order_date DATE")

In [0]:
deltaTable2.history().display()

In [0]:
display(dbutils.fs.ls(output_path + "parquet2"))

#### Converting Partitioned Parquet Dataset to Delta using SQL

In [0]:
%fs ls /Volumes/workspace/default/data/output/parquet3

In [0]:
%sql
CONVERT TO DELTA PARQUET.`/Volumes/workspace/default/data/output/parquet3`
PARTITIONED BY (order_date DATE)

In [0]:
%fs ls /Volumes/workspace/default/data/output/parquet3

#### Cleanup

In [0]:
dbutils.fs.rm(output_path, True)